# Extração Sakila → Bronze

Carrega as tabelas do banco `sakila` (Postgres) para o data lake
(catalogo `nessie`/Iceberg, schema `bronze_sakila`), uma tabela por
tabela.

A escrita é `overwrite`: reexecutar o notebook reextrai os dados
(sem duplicar). Rode as células em ordem.

In [ ]:
SPARK_APP_NAME = "extrai-sakila-bronze"

from nucleo.spark import (
    POSTGRES_PASSWORD,
    POSTGRES_URL,
    POSTGRES_USER,
    cria_spark_session,
)

spark = cria_spark_session(SPARK_APP_NAME)
spark

In [ ]:
SCHEMA = "bronze_sakila"

TABELAS = [
    "actor",
    "address",
    "category",
    "city",
    "country",
    "customer",
    "film",
    "film_actor",
    "film_category",
    "inventory",
    "language",
    "payment",
    "rental",
    "staff",
    "store",
]

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS nessie.{SCHEMA}")

for tabela in TABELAS:
    df = (
        spark.read.format("jdbc")
        .option("url", POSTGRES_URL)
        .option("user", POSTGRES_USER)
        .option("password", POSTGRES_PASSWORD)
        .option("dbtable", tabela)
        .load()
    )
    # createOrReplace: reexecucao do notebook reextrai sem duplicar dados
    df.writeTo(f"nessie.{SCHEMA}.{tabela}").createOrReplace()

In [ ]:
for tabela in TABELAS:
    n = spark.table(f"nessie.{SCHEMA}.{tabela}").count()
    print(f"{SCHEMA}.{tabela}: {n} linhas")